In [57]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms
import torchinfo
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torch.utils.data import Dataset

import helper_utils


In [58]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using device: CUDA")
else:
    device = torch.device("cpu")
    print(f"Using device: CPU")

Using device: CUDA


## Hyperparameters

In [59]:
BATCH_SIZE = 32

## Datasets and DataLoaders

### Dataset Transforms

In [60]:
train_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5] * 3, std=[0.5] * 3)
])

val_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5] * 3, std=[0.5] * 3)
])

### Train, Test and Validation Datasets and DataLoaders

In [61]:
train_dir = os.path.join("data", "train")
test_dir = os.path.join("data", "test")
val_dir = os.path.join("data", "validation")

train_dataset = ImageFolder(root = train_dir, transform = train_transforms)
val_dataset = ImageFolder(root = val_dir, transform = val_transforms)
test_dataset = ImageFolder(root = test_dir, transform = val_transforms)

train_loader = DataLoader(train_dataset, batch_size = BATCH_SIZE, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = BATCH_SIZE, shuffle = False)
test_loader = DataLoader(test_dataset, batch_size = BATCH_SIZE, shuffle = False)

In [62]:
classes = train_dataset.classes

num_classes = len(classes)

print(f"Classes: {classes}")
print(f"Number of classes: {num_classes}")

Classes: ['dress', 'hat', 'longsleeve', 'outwear', 'pants', 'shirt', 'shoes', 'shorts', 'skirt', 't-shirt']
Number of classes: 10


In [63]:
class InvertedResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride, expansion_factor, shortcut = None):
        super(InvertedResidualBlock, self).__init__()

        hidden_dim = in_channels * expansion_factor



        # 1 x 1 pointwise convolution to expand the number of channels
        self.expand = nn.Sequential(
            nn.Conv2d(in_channels =  in_channels,
                      out_channels = hidden_dim,
                      kernel_size = 1, 
                      bias = False
                      ),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU(inplace = True)
        )

        # 3 x 3 depthwise convolution
        self.depthwise = nn.Sequential(
            nn.Conv2d(in_channels = hidden_dim,
                      out_channels = hidden_dim,
                      kernel_size = 3,
                      stride = stride,
                      padding = 1,
                      bias = False,
                      groups = hidden_dim
                      ),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU(inplace = True)
        )

        # 1 x 1 pointwise convolution to project back to the original number of channels
        # Bottleneck
        self.project = nn.Sequential(
            nn.Conv2d(in_channels = hidden_dim,
                      out_channels = out_channels,
                      kernel_size = 1,
                      bias = False),
            nn.BatchNorm2d(out_channels)
        )

        # Optional shortcut connection for residual learning
        self.shortcut = shortcut



    def forward(self, x):

        skip = x

        out = self.expand(x)
        out = self.depthwise(out)
        out = self.project(out)
        
        if self.shortcut is not None:
            skip = self.shortcut(x)

        if skip.shape == out.shape:
            out = out + skip

        return F.relu(out)



In [64]:
class MobileNetBackbone(nn.Module):
    def __init__(self):

        super(MobileNetBackbone, self).__init__()


        self.stem = nn.Sequential(
           nn.Conv2d(in_channels = 3, out_channels = 16, kernel_size = 3,stride = 2, padding = 1, bias = False),
           nn.BatchNorm2d(16),
           nn.ReLU(inplace = True)
        )

        self.blocks = nn.Sequential(
            self._make_block(in_channels = 16, out_channels = 24, stride = 2, expansion_factor = 3),
            self._make_block(in_channels = 24, out_channels = 32, stride = 2, expansion_factor = 3),
            self._make_block(in_channels = 32, out_channels = 64, stride = 2, expansion_factor = 6),
        )





    def _make_block(self, in_channels, out_channels, stride, expansion_factor):

        condition = (stride !=1) or (in_channels != out_channels)

        if condition:
            shortcut = nn.Sequential(
                nn.Conv2d(in_channels = in_channels,
                          out_channels = out_channels,
                          kernel_size = 1,
                          stride = stride,
                          padding = 0,
                          bias = False),
                nn.BatchNorm2d(out_channels)
            )

        else:
            shortcut = None


        block = InvertedResidualBlock(
            in_channels = in_channels,
            out_channels = out_channels,
            stride = stride,
            expansion_factor = expansion_factor,
            shortcut = shortcut
        )

        return block

    def forward(self, x):
        x = self.stem(x)

        x = self.blocks(x)

        return x


In [65]:
# --- Verification ---
# Define parameters for verification
batch_size=32
img_size = 64 # Input image height/width
depth = 3 # Summary depth

# Instantiate the backbone
backbone = MobileNetBackbone()

# Define the input tensor shape
input_size = (batch_size, 3, img_size, img_size)

# Configuration for torchinfo summary
config = {
    "input_size": input_size,
    "attr_names": ["input_size", "output_size", "num_params"],
    "col_names_display": ["Input Shape ", "Output Shape", "Param #"],
    "depth": depth
}

# Generate the summary
summary = torchinfo.summary(
    model=backbone,
    input_size=config["input_size"],
    col_names=config["attr_names"],
    depth=config["depth"]
)

# Display the formatted summary
print("--- Backbone Summary ---\n")
helper_utils.display_torch_summary(summary, config["attr_names"], config["col_names_display"], config["depth"])

--- Backbone Summary ---



Layer (type (var_name):depth-idx),Input Shape,Output Shape,Param #
MobileNetBackbone (MobileNetBackbone),"[32, 3, 64, 64]","[32, 64, 4, 4]",--
Sequential (stem): 1-1,"[32, 3, 64, 64]","[32, 16, 32, 32]",--
Conv2d (0): 2-1,"[32, 3, 64, 64]","[32, 16, 32, 32]",432
BatchNorm2d (1): 2-2,"[32, 16, 32, 32]","[32, 16, 32, 32]",32
ReLU (2): 2-3,"[32, 16, 32, 32]","[32, 16, 32, 32]",--
Sequential (blocks): 1-2,"[32, 16, 32, 32]","[32, 64, 4, 4]",--
InvertedResidualBlock (0): 2-4,"[32, 16, 32, 32]","[32, 24, 16, 16]",--
Sequential (expand): 3-1,"[32, 16, 32, 32]","[32, 48, 32, 32]",864
Sequential (depthwise): 3-2,"[32, 48, 32, 32]","[32, 48, 16, 16]",528
Sequential (project): 3-3,"[32, 48, 16, 16]","[32, 24, 16, 16]","1,200"


In [66]:
class MobileNetLikeClassifier(nn.Module):
    def __init__(self, num_classes):

        super(MobileNetLikeClassifier, self).__init__()

        self.backbone = MobileNetBackbone()

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, num_classes)
        )


    def forward(self, x):
        x = self.backbone(x)

        x = self.head(x)

        return x

In [67]:
mobilenet_classifier = MobileNetLikeClassifier(num_classes=num_classes)

In [68]:
# Define parameters for verification
batch_size=32
img_size = 64 # Input image height/width
depth = 3 # Summary depth

# Define the input tensor shape
input_size = (batch_size, 3, img_size, img_size)

# Configuration for torchinfo summary
config = {
    "input_size": input_size,
    "attr_names": ["input_size", "output_size", "num_params"],
    "col_names_display": ["Input Shape ", "Output Shape", "Param #"],
    "depth": depth # Show layers up to 3 levels deep for detail
}

# Generate the summary for the complete classifier
summary = torchinfo.summary(
    model=mobilenet_classifier,
    input_size=config["input_size"],
    col_names=config["attr_names"],
    depth=config["depth"]
)

# Display the formatted summary
print("--- Classifier Summary ---\n")
helper_utils.display_torch_summary(summary, config["attr_names"], config["col_names_display"], config["depth"])

--- Classifier Summary ---



Layer (type (var_name):depth-idx),Input Shape,Output Shape,Param #
MobileNetLikeClassifier (MobileNetLikeClassifier),"[32, 3, 64, 64]","[32, 10]",--
MobileNetBackbone (backbone): 1-1,"[32, 3, 64, 64]","[32, 64, 4, 4]",--
Sequential (stem): 2-1,"[32, 3, 64, 64]","[32, 16, 32, 32]",--
Conv2d (0): 3-1,"[32, 3, 64, 64]","[32, 16, 32, 32]",432
BatchNorm2d (1): 3-2,"[32, 16, 32, 32]","[32, 16, 32, 32]",32
ReLU (2): 3-3,"[32, 16, 32, 32]","[32, 16, 32, 32]",--
Sequential (blocks): 2-2,"[32, 16, 32, 32]","[32, 64, 4, 4]",--
InvertedResidualBlock (0): 3-4,"[32, 16, 32, 32]","[32, 24, 16, 16]","3,024"
InvertedResidualBlock (1): 3-5,"[32, 24, 16, 16]","[32, 32, 8, 8]","5,864"
InvertedResidualBlock (2): 3-6,"[32, 32, 8, 8]","[32, 64, 4, 4]","23,232"
